In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

## Read in the data

In [2]:
# read in the data
hmda_2020 = pd.read_csv('../data/msamd_12060_2020.csv') 
hmda_2021 = pd.read_csv('../data/msamd_12060_2021.csv')
hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv') 
hmda_2023 = pd.read_csv('../data/msamd_12060_2023.csv')
hmda_2024 = pd.read_csv('../data/msamd_12054_2024.csv')

C:\Users\Cat\AppData\Local\Temp\ipykernel_26116\211953371.py:2: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2020 = pd.read_csv('../data/msamd_12060_2020.csv')
C:\Users\Cat\AppData\Local\Temp\ipykernel_26116\211953371.py:3: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2021 = pd.read_csv('../data/msamd_12060_2021.csv')
C:\Users\Cat\AppData\Local\Temp\ipykernel_26116\211953371.py:4: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv')
C:\Users\Cat\AppData\Local\Temp\ipykernel_26116\211953371.py:5: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_202

#### concatenate hmda for 2020-2024 together

In [3]:
hmda_2020['year'] = 2020
hmda_2021['year'] = 2021
hmda_2022['year'] = 2022
hmda_2023['year'] = 2023
hmda_2024['year'] = 2024

hmda_all = pd.concat([hmda_2020, hmda_2021, hmda_2022, hmda_2023, hmda_2024], ignore_index=True)

#### create a unique ID column for all rows of the data

In [4]:
hmda_all['unique_id'] = range(len(hmda_all))

#### count number of rows of full dataset before filtering --> 2,046,788

In [5]:
hmda_all[hmda_all.columns[0]].count()

np.int64(2046788)

##### Remove all values that do not reflect approved and accepted/originated = 1, or denied = 3

In [6]:
# remove all values that do not reflect approved, denied, or approved but not accepted
# Description: The action taken on the covered loan or application
# Values:
# 1 - Loan originated
# 2 - Application approved but not accepted
# 3 - Application denied
# 4 - Application withdrawn by applicant
# 5 - File closed for incompleteness
# 6 - Purchased loan
# 7 - Preapproval request denied
# 8 - Preapproval request approved but not accepted

list_of_values_to_keep = [1,3]

hmda_all = hmda_all[hmda_all['action_taken'].isin(list_of_values_to_keep)]
hmda_all['action_taken'].unique()

array([1, 3])

##### change name of column

In [7]:
hmda_all['approved_originated_or_denied'] = hmda_all['action_taken']

##### Convert 1s and 3s to 1s and 0s

In [8]:
hmda_all['approved_originated_or_denied'] = hmda_all['approved_originated_or_denied'].replace({1 : 1, 3 : 0})

#### count number of rows that were approved originated or denied --> 1,340,915

In [9]:
# number of non-NaN values in first column
hmda_all[hmda_all.columns[0]].count()

np.int64(1340915)

##### Remove not applicable (could include applicants with little to no credit and exempt institutions) to eliminate legal entities like corporations etc from being included in the data instead of individuals

In [10]:
# remove not applicable (could include applicants with little to no credit) and exempt institutions
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8]

hmda_all = hmda_all[hmda_all['applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_all['applicant_credit_score_type'].unique()

array([2, 3, 1, 7, 4, 8, 6, 5])

#### count number of rows that were left after NA and Exempt were removed --> 1,169,761

In [11]:
# number of non-NaN values in first column
hmda_all[hmda_all.columns[0]].count()

np.int64(1169761)

##### Remove not applicable (could include applicants with little to no credit and exempt institutions) to eliminate legal entities like corporations etc from being included in the data instead of individuals`

In [12]:
# remove not applicable (could include applicants with little to no credit and exempt institutions)
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 10 - No co-applicant
# 11 – FICO Score 9
# 12 – FICO Score 8
# 13 – FICO Score 10
# 14 – FICO Score 10T
# 15 - VantageScore 4.0
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8,10,11,12,13,14,15]

hmda_all = hmda_all[hmda_all['co-applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_all['co-applicant_credit_score_type'].unique()

array([10,  1,  2,  3,  7,  4,  8,  5,  6, 11, 14])

#### count number of rows that were left after NA and Exempt were removed from co-applicant --> 942,372

In [13]:
# number of non-NaN values in first column
hmda_all[hmda_all.columns[0]].count()

np.int64(942372)

## Convert financial information to numeric

##### Coerce to numeric the loan_to_value_ratio 

In [14]:
print(hmda_all['loan_to_value_ratio'].dtype)
hmda_all['loan_to_value_ratio'] = pd.to_numeric(hmda_all['loan_to_value_ratio'], errors='coerce')
print(hmda_all['loan_to_value_ratio'].dtype)

object
float64


##### Coerce to numeric the property_value 

In [15]:
print(hmda_all['property_value'].dtype)
hmda_all['property_value'] = pd.to_numeric(hmda_all['property_value'], errors='coerce')
print(hmda_all['property_value'].dtype)

object
float64


##### Check data type of income

In [16]:
print(hmda_all['income'].dtype)

float64


##### Coerce to numeric the interest_rate 

In [17]:
print(hmda_all['interest_rate'].dtype)
hmda_all['interest_rate'] = pd.to_numeric(hmda_all['interest_rate'], errors='coerce')
print(hmda_all['interest_rate'].dtype)

object
float64


##### Coerce to numeric the rate_spread 

In [18]:
print(hmda_all['rate_spread'].dtype)
hmda_all['rate_spread'] = pd.to_numeric(hmda_all['rate_spread'], errors='coerce')
print(hmda_all['rate_spread'].dtype)

object
float64


## Scale appropriate financial information

#### Scale ffiec_msa_md_median_family_income

In [19]:
hmda_all['ffiec_msa_md_median_family_income_scaled'] = hmda_all['ffiec_msa_md_median_family_income'] / 1000

#### Scale loan_amount

In [20]:
hmda_all['loan_amount_scaled'] = hmda_all['loan_amount'] / 1000

#### Scale property_value

In [21]:
hmda_all['property_value_scaled'] = hmda_all['property_value'] / 1000

## Converting categorical data into numeric for the purpose of running a logistic regression

### AUS Automated Underwriting System

#### Melt the AUS columns into a smaller df to OHE those columns

In [24]:
# melting data into smaller df to ohe it
hmda_all_melted = hmda_all.melt(
    id_vars="unique_id", 
    value_vars=["aus-1", "aus-2", "aus-3", "aus-4", "aus-5"],
    var_name="aus_types", 
    value_name="aus_values"
) 

##### Fill in NaNs with 0s and convert to an integer from float

In [25]:
# fill NaNs with zeroes and change the data type to integer instead of float
hmda_all_melted["aus_values"] = hmda_all_melted["aus_values"].fillna(0).astype(int)

#### One Hot Encode data

In [26]:
# use OneHotEncoder
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_all_melted[["aus_values"]])

# combine the encoded columns back with the original df
hmda_all_melted = pd.concat([hmda_all_melted.drop(columns=["aus_values"]), encoded_df], axis=1)

##### Drop unnecessary columns 

In [27]:
hmda_all_melted = hmda_all_melted.drop(columns=["aus_values_0", "aus_types"])

##### Group by the unique_id column, add those values together, and reset the index

In [28]:
# group by the unique_id column, sum those values together, and reset the index
hmda_all_melted = hmda_all_melted.groupby("unique_id").sum().reset_index()

##### make aus_columns integers instead of floats

In [29]:
# make aus_columns integers instead of floats
aus_columns = ['aus_values_1', 'aus_values_2', 'aus_values_3', 'aus_values_4', 'aus_values_5', 'aus_values_6']

hmda_all_melted[aus_columns] = hmda_all_melted[aus_columns].fillna(0).astype(int)

#### Merge the AUS df and the original HMDA df together